In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt


In [47]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (140184, 39)
Val:   (26307, 39)
Test:  (35034, 39)


In [48]:
feature_type_summary = pd.DataFrame(
    {
        "feature": X_train.columns,
        "dtype": X_train.dtypes.astype(str).values,
    }
)

pd.set_option("display.max_rows", None)
print(feature_type_summary.sort_values("dtype"))


                          feature    dtype
39          AttachedGarageYN_True     bool
37             PoolPrivateYN_True     bool
36          PoolPrivateYN_Missing     bool
35                    ViewYN_True     bool
34                 ViewYN_Missing     bool
38       AttachedGarageYN_Missing     bool
17                    LotSizeArea  float64
16                   LotSizeAcres  float64
15              LotSizeSquareFeet  float64
14                 AssociationFee  float64
13                   ParkingTotal  float64
12                   GarageSpaces  float64
0                    City_encoded  float64
10                      YearBuilt  float64
9           BathroomsTotalInteger  float64
8                   BedroomsTotal  float64
7                      LivingArea  float64
6                       Longitude  float64
1              PostalCode_encoded  float64
5                        Latitude  float64
2          CountyOrParish_encoded  float64
3            MLSAreaMajor_encoded  float64
4      High

## Normalize numerical features

In [49]:
bool_cols = X_train.select_dtypes(include=["bool"]).columns.tolist()
missing_flag_cols = [c for c in X_train.columns if c.endswith("_missing")]

no_scale_cols = list(set(bool_cols + missing_flag_cols))
scale_cols = [c for c in X_train.columns if c not in no_scale_cols]

print(f"Columns to scale: {len(scale_cols)}")
print(scale_cols)
print(f"\nColumns NOT to scale (binary flags/dummies): {len(no_scale_cols)}")
print(no_scale_cols)


Columns to scale: 18
['City_encoded', 'PostalCode_encoded', 'CountyOrParish_encoded', 'MLSAreaMajor_encoded', 'HighSchoolDistrict_encoded', 'Latitude', 'Longitude', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt', 'Stories', 'GarageSpaces', 'ParkingTotal', 'AssociationFee', 'LotSizeSquareFeet', 'LotSizeAcres', 'LotSizeArea']

Columns NOT to scale (binary flags/dummies): 22
['Latitude_missing', 'ParkingTotal_missing', 'AssociationFee_missing', 'PoolPrivateYN_missing', 'ViewYN_True', 'ViewYN_Missing', 'HighSchoolDistrict_missing', 'BathroomsTotalInteger_missing', 'MLSAreaMajor_missing', 'LotSizeSquareFeet_missing', 'Stories_missing', 'AttachedGarageYN_True', 'AttachedGarageYN_Missing', 'ViewYN_missing', 'PoolPrivateYN_Missing', 'LotSizeArea_missing', 'YearBuilt_missing', 'GarageSpaces_missing', 'AttachedGarageYN_missing', 'LivingArea_missing', 'PoolPrivateYN_True', 'LotSizeAcres_missing']


In [50]:
exclude_cols = ["log_price", "ClosePrice"]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df["log_price"]
y_val = val_df["log_price"]
y_test = test_df["log_price"]

print(f"Feature count: {len(feature_cols)}")


Feature count: 37


In [51]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val_scaled[scale_cols] = scaler.transform(X_val[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print("Scaling done (binary/flag columns left unscaled).")
print(X_train_scaled[scale_cols].describe().loc[["mean", "std"]])


Scaling done (binary/flag columns left unscaled).
      City_encoded  PostalCode_encoded  CountyOrParish_encoded  \
mean  2.690842e-15        3.295024e-15            1.573307e-15   
std   1.000004e+00        1.000004e+00            1.000004e+00   

      MLSAreaMajor_encoded  HighSchoolDistrict_encoded      Latitude  \
mean          1.904999e-15               -3.933267e-16  3.959219e-15   
std           1.000004e+00                1.000004e+00  1.000004e+00   

         Longitude    LivingArea  BedroomsTotal  BathroomsTotalInteger  \
mean  1.536002e-15 -3.325030e-17   1.111047e-16          -2.027457e-17   
std   1.000004e+00  1.000004e+00   1.000004e+00           1.000004e+00   

         YearBuilt       Stories  GarageSpaces  ParkingTotal  AssociationFee  \
mean  2.816138e-15  1.102937e-16  4.176562e-17  4.054915e-19    4.784800e-17   
std   1.000004e+00  1.000004e+00  1.000004e+00  1.000004e+00    1.000004e+00   

      LotSizeSquareFeet  LotSizeAcres   LotSizeArea  
mean      -8.109

## Linear Regression

In [52]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [53]:
import statsmodels.api as sm

X_train_sm = sm.add_constant(X_train_scaled.astype(float))
ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.883
Model:                            OLS   Adj. R-squared:                  0.883
Method:                 Least Squares   F-statistic:                 2.846e+04
Date:                Wed, 15 Jul 2026   Prob (F-statistic):               0.00
Time:                        00:08:39   Log-Likelihood:                 9167.9
No. Observations:              140184   AIC:                        -1.826e+04
Df Residuals:                  140146   BIC:                        -1.789e+04
Df Model:                          37                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

In [58]:
coef_summary = pd.DataFrame(
    {
        "coefficient": ols_model.params,
        "abs_coefficient": ols_model.params.abs(),
        "p.value": ols_model.pvalues,
    }
)

coef_summary = coef_summary.drop("const")


def sig_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    elif p < 0.1:
        return "."
    return ""


coef_summary["sig"] = coef_summary["p.value"].apply(sig_stars)
coef_summary = coef_summary.sort_values("abs_coefficient", ascending=False)

pd.set_option("display.float_format", "{:.4f}".format)
print(coef_summary)


                               coefficient  abs_coefficient  p.value  sig
LivingArea_missing                  0.5422           0.5422   0.0000  ***
PostalCode_encoded                  0.3198           0.3198   0.0000  ***
LivingArea                          0.1975           0.1975   0.0000  ***
YearBuilt_missing                   0.1441           0.1441   0.0000  ***
LotSizeSquareFeet_missing           0.1235           0.1235   0.3941     
BathroomsTotalInteger_missing      -0.1143           0.1143   0.0184    *
Latitude                           -0.1069           0.1069   0.0000  ***
Longitude                          -0.1061           0.1061   0.0000  ***
LotSizeAcres_missing               -0.1034           0.1034   0.0935    .
City_encoded                        0.1027           0.1027   0.0000  ***
Latitude_missing                    0.0878           0.0878   0.2737     
PoolPrivateYN_True                  0.0681           0.0681   0.0000  ***
PoolPrivateYN_Missing               0.

In [54]:
y_pred_test = model.predict(X_test_scaled)

r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

print(f"Test R²:   {r2_test:.4f}")
print(f"Test RMSE (log scale): {rmse_test:.4f}")
print(f"Test MAE  (log scale): {mae_test:.4f}")


Test R²:   0.8804
Test RMSE (log scale): 0.2343
Test MAE  (log scale): 0.1659


In [55]:
y_pred_val = model.predict(X_val_scaled)
r2_val = r2_score(y_val, y_pred_val)
print(f"Validation R²: {r2_val:.4f}")


Validation R²: 0.8733


In [56]:
baseline_results = pd.DataFrame(
    [
        {
            "model": "Linear Regression",
            "val_R2": r2_val,
            "test_R2": r2_test,
            "test_RMSE_log": rmse_test,
            "test_MAE_log": mae_test,
        }
    ]
)

print(baseline_results)
baseline_results.to_csv("baseline_results.csv", index=False)


               model    val_R2  test_R2  test_RMSE_log  test_MAE_log
0  Linear Regression  0.873322  0.88041       0.234263       0.16594
